Imports

In [10]:
import numpy as np
import pandas as pd

Intent detector

In [11]:
def detect_intent(user_query):
    query = user_query.lower()

    eligibility_keywords = [
        "eligible", "can i donate", "allowed to donate",
        "hemoglobin", "weight", "medical condition",
        "sick", "illness", "condition"
    ]

    availability_keywords = [
        "available", "likely to donate", "donate again",
        "return", "campaign", "invite", "reminder"
    ]

    if any(word in query for word in eligibility_keywords) and any(word in query for word in availability_keywords):
        return "combined"

    if any(word in query for word in eligibility_keywords):
        return "eligibility"

    if any(word in query for word in availability_keywords):
        return "availability"

    return "general_qa"

Load Eligibility engine rules

In [12]:
import json

with open("eligibility_rules.json", "r") as file:
    eligibility_rules = json.load(file)

Eligibility rule engine

In [13]:
def eligibility_engine(donor, rules):

    condition = donor.get("medical_condition", "None")
    weight = donor.get("weight_kg")
    hemoglobin = donor.get("hemoglobin_g_dl")

    disallowed_conditions = rules["disallowed_conditions"]
    min_weight = rules["min_weight_kg"]
    min_hemoglobin = rules["min_hemoglobin_g_dl"]

    if condition in disallowed_conditions:
        return {
            "eligible": False,
            "reason": f"Medical condition '{condition}' prevents donation.",
            "decision": "Not Eligible"
        }

    if weight is not None and weight < min_weight:
        return {
            "eligible": False,
            "reason": f"Weight below minimum requirement ({min_weight} kg).",
            "decision": "Not Eligible"
        }

    if hemoglobin is not None and hemoglobin < min_hemoglobin:
        return {
            "eligible": False,
            "reason": f"Hemoglobin below minimum requirement ({min_hemoglobin} g/dL).",
            "decision": "Not Eligible"
        }

    return {
        "eligible": True,
        "reason": "Donor satisfies eligibility requirements.",
        "decision": "Eligible"
    }

Availability prediction wrapper

In [14]:
def availability_engine(donor, availability_model):
    """
    donor should contain:
    recency, frequency, time
    """

    features = pd.DataFrame([{
        "Recency": donor.get("recency"),
        "Frequency": donor.get("frequency"),
        "Time": donor.get("time")
    }])

    probability = availability_model.predict_proba(features)[0][1]
    prediction = availability_model.predict(features)[0]

    return {
        "available": bool(prediction == 1),
        "probability": round(probability, 4),
        "decision": "Likely Available" if prediction == 1 else "Not Likely Available"
    }

Combined decision logic

In [15]:
def combined_decision(eligibility_result, availability_result):
    eligible = eligibility_result["eligible"]
    available = availability_result["available"]

    if eligible and available:
        action = "Invite donor for the next campaign."
    elif eligible and not available:
        action = "Send soft reminder or awareness message."
    elif not eligible and available:
        action = "Do not invite yet. Provide medical guidance first."
    else:
        action = "Low priority for campaign invitation."

    return {
        "eligibility": eligibility_result["decision"],
        "availability": availability_result["decision"],
        "availability_probability": availability_result["probability"],
        "reason": eligibility_result["reason"],
        "recommended_action": action
    }

Final response formatter

In [16]:
def format_response(result):
    response = f"""
Eligibility Status: {result['eligibility']}
Availability Status: {result['availability']}
Availability Probability: {result['availability_probability']}

Reason:
{result['reason']}

Recommended Action:
{result['recommended_action']}
"""
    return response

## load the final availability Logistic Regression model

In [17]:
import joblib

availability_model = joblib.load("availability_logistic_model.pkl")

Test of the full pipeline

In [18]:
sample_donor = {
    "medical_condition": "None",
    "weight_kg": 62,
    "hemoglobin_g_dl": 13.1,
    "recency": 2,
    "frequency": 8,
    "time": 24
}

eligibility_result = eligibility_engine(sample_donor, eligibility_rules)
availability_result = availability_engine(sample_donor, availability_model)

final_result = combined_decision(
    eligibility_result,
    availability_result
)

print(format_response(final_result))


Eligibility Status: Eligible
Availability Status: Likely Available
Availability Probability: 0.734

Reason:
Donor satisfies eligibility requirements.

Recommended Action:
Invite donor for the next campaign.

